# Индивидуальная часть проекта — *Caenorhabditis brenneri*

**Сборка:** `GCA_964036135.1_CAEBRE_CFB2252`, GenBank, штамм **CFB2252**  
**Организм:** *Caenorhabditis brenneri* - свободноживущая нематода.

| Параметр | Значение |
|---|---|
| Уровень сборки | Chromosome |
| GC-состав | ~37 % |
| Аннотация NCBI | **есть** (GTF + протеом скачиваем с FTP) |
| Среда обитания | почва / гниющая органика, ~20 °C |


## Что и откуда скачиваем (NCBI FTP `GCA_964036135.1_CAEBRE_CFB2252`)
1. **`*_genomic.fna.gz`** - геном
2. **`*_protein.faa.gz`** - протеом
3. **`*_genomic.gtf.gz`** - аннотация генов

## План
0. Установка инструментов
1. Скачивание данных
2. Статистика генома  
3. Эпигенетические гены (Pfam/HMMER)
4. G-квадруплексы
5. Z-ДНК (zhunt)  
6. Z-ДНК (ZDNABERT)  
7. Распределение структур по геному
8. Сравнение с фоном
9. Итоги



## 0. Установка инструментов



In [1]:
%pip install -q pandas openpyxl

In [2]:
import os

In [3]:
%%bash
apt-get update -qq
apt-get install -y -qq bedtools
apt-get install -y -qq hmmer
bedtools --version
hmmsearch -h | head -2

Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 118296 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64.deb ...
Unpacking hmmer (3.3.2+dfsg-1) ...
Setting up libdivsufsort3:amd64 (2.0.1-5) ...
Setting up hmmer (3.3.2+dfsg-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/l

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%bash
# если инструменты уже есть — ничего не делаем
if command -v bedtools >/dev/null && command -v hmmsearch >/dev/null; then
  echo "инструменты на месте"
else
  mamba install -y -c bioconda -c conda-forge bedtools hmmer
fi

инструменты на месте


In [5]:
from pathlib import Path
from collections import defaultdict, Counter
import re, subprocess
import pandas as pd

GENOME = "data/genome.fna"
GTF = "data/annotation.gtf"
PROT = "data/proteome.faa"
SIZES = "data/genome.sizes"
# TEST_MODE=True
TEST_MODE = False
TEST_CHROMS = []

## 1. Скачивание данных *Caenorhabditis brenneri*

У этой сборки есть готовый протеом и GTF-аннотация — скачиваем все три файла напрямую с NCBI FTP.


In [6]:
%%bash
mkdir -p data

# Протеом
[ -s data/proteome.faa ] || {
  wget -q https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_protein.faa.gz -O data/proteome.faa.gz
  gunzip -q -f data/proteome.faa.gz
}

# Геном
[ -s data/genome.fna ] || {
  wget -q https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.fna.gz -O data/genome.fna.gz
  gunzip -q -f data/genome.fna.gz
}
echo "Размер генома (нуклеотидов):"
grep -v "^>" data/genome.fna | tr -d '\n' | wc -c

# Аннотация
[ -s data/annotation.gtf ] || {
  wget -q https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/964/036/135/GCA_964036135.1_CAEBRE_CFB2252/GCA_964036135.1_CAEBRE_CFB2252_genomic.gtf.gz -O data/annotation.gtf.gz
  gunzip -q -f data/annotation.gtf.gz
}
echo "Число аннотированных записей:"
grep -v "^#" data/annotation.gtf | wc -l

Размер генома (нуклеотидов):
126489859
Число аннотированных записей:
390478


In [7]:
# Быстрая проверка скачанных файлов
import os
for f in [GENOME, GTF, PROT]:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f"{'OK' if size > 0 else 'MISSING':6} {f}  ({size:,} bytes)")
print(f"Белков в протеоме: {sum(1 for l in open(PROT) if l.startswith('>')):,}")

OK     data/genome.fna  (128,092,971 bytes)
OK     data/annotation.gtf  (140,624,396 bytes)
OK     data/proteome.faa  (12,208,306 bytes)
Белков в протеоме: 25,588


## 2. Статистика генома (для отчёта)

Длина, GC, доля N, N50, число скаффолдов.


In [8]:
def read_fasta(path):
    seqs, name, buf = {}, None, []
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                if name: seqs[name] = "".join(buf)
                name = line[1:].split()[0]; buf = []
            else:
                buf.append(line.strip())
        if name: seqs[name] = "".join(buf)
    return seqs

In [9]:
genome = {k: v.upper() for k, v in read_fasta(GENOME).items()}
CHROM_SIZES = {k: len(v) for k, v in genome.items()}

if not TEST_CHROMS:
    TEST_CHROMS = [list(genome.keys())[0]]

WORK_CHROMS = TEST_CHROMS if TEST_MODE else list(genome.keys())

total = sum(CHROM_SIZES.values())
gc = sum(s.count("G")+s.count("C") for s in genome.values())
nN = sum(s.count("N") for s in genome.values())
lens = sorted(CHROM_SIZES.values(), reverse=True)
run, n50 = 0, 0
for L in lens:
    run += L
    if run >= total/2: n50 = L; break

In [10]:
print(f"Последовательностей : {len(genome)}")
print(f"Длина генома: {total:,} bp")
print(f"GC-состав: {100*gc/(total-nN):.2f} %")
print(f"Доля N (gaps): {100*nN/total:.4f} %  ({nN:,} bp)")
print(f"N50: {n50:,} bp")
for k,v in sorted(CHROM_SIZES.items(), key=lambda x:-x[1])[:20]:
    print(f"   {k:25} {v:>12,} bp")

# genome.sizes - для bedtools
with open(SIZES, "w") as f:
    for k in WORK_CHROMS:
        f.write(f"{k}\t{CHROM_SIZES[k]}\n")
print("\nрабочих хромосом (WORK_CHROMS):", len(WORK_CHROMS), "| TEST_MODE =", TEST_MODE)

Последовательностей : 170
Длина генома: 126,489,859 bp
GC-состав: 38.42 %
Доля N (gaps): 0.0000 %  (0 bp)
N50: 20,836,669 bp
   OZ038411.1                  28,009,396 bp
   OZ038409.1                  24,531,178 bp
   OZ038410.1                  20,836,669 bp
   OZ038407.1                  17,865,897 bp
   OZ038408.1                  15,980,374 bp
   OZ038406.1                  15,949,836 bp
   CAXIWG010000135.1              179,310 bp
   CAXIWG010000134.1              125,371 bp
   CAXIWG010000130.1              124,928 bp
   CAXIWG010000129.1              124,565 bp
   CAXIWG010000123.1              124,243 bp
   CAXIWG010000132.1              124,037 bp
   CAXIWG010000120.1              123,294 bp
   CAXIWG010000124.1              122,996 bp
   CAXIWG010000136.1              119,076 bp
   CAXIWG010000103.1              112,305 bp
   CAXIWG010000122.1              103,281 bp
   CAXIWG010000150.1               68,670 bp
   CAXIWG010000137.1               54,632 bp
   CAXIWG010000138.1

## 2.5. Подготовка аннотации

Аннотация скачана с NCBI (GTF).


In [11]:
# Проверяем, какие feature-типы есть в GTF
import subprocess
result = subprocess.run(
    ["awk", "$1!~/^#/{print $3}", GTF],
    capture_output=True, text=True
)
from collections import Counter
types = Counter(result.stdout.strip().split("\n"))
print("Feature types in GTF:")
for t, n in sorted(types.items(), key=lambda x: -x[1])[:15]:
    print(f"{t:11} {n:>10,}")

Feature types in GTF:
exon           144,861
CDS            143,339
gene            25,612
transcript      25,612
stop_codon      25,554
start_codon     25,500


In [12]:
# Число генов в аннотации
n_genes = sum(1 for l in open(GTF)
              if not l.startswith("#") and l.split("\t")[2:3] == ["gene"])
n_prot  = sum(1 for l in open(PROT) if l.startswith(">"))
print(f"Генов в GTF: {n_genes:,}")
print(f"Белков в PROT: {n_prot:,}")

Генов в GTF: 25,612
Белков в PROT: 25,588


## 3. Эпигенетические гены: поиск ≥10 семейств по Pfam/HMMER

протеом -> `hmmsearch` против профилей выбранных Pfam-семейств ->
сопоставляем найденные белки с генами (через GFF) -> таблица «семейство–ген».

Профили Pfam тянем поштучно из InterPro (`…?annotation=hmm`).


In [13]:
FAMILIES = [
 ("PF00145","DNA_methylase", "ДНК-метилтрансферазы C5 (DNMT1/3)", "DNA methylation (write)"),
 ("PF01429","MBD", "Метил-CpG-связывающий домен (MeCP2/MBD)", "DNA methylation (read)"),
 ("PF02008","zf-CXXC", "CXXC цинк-палец (связывание неметил-CpG)", "DNA/CpG binding"),
 ("PF12851","Tet_JBP", "TET/JBP диоксигеназы (деметилирование)", "DNA demethylation"),
 ("PF00856","SET", "SET гистон-метилтрансферазы (EZH2/SETD)", "Histone methylation (write)"),
 ("PF02373","JmjC", "JmjC гистон-деметилазы (KDM)", "Histone demethylation (erase)"),
 ("PF00850","Hist_deacetyl", "Гистон-деацетилазы HDAC I/II", "Histone deacetylation (erase)"),
 ("PF02146","SIR2", "Сиртуины - HDAC класс III", "Histone deacetylation (erase)"),
 ("PF01853","MOZ_SAS", "MYST гистон-ацетилтрансферазы", "Histone acetylation (write)"),
 ("PF00583","Acetyltransf_1", "GNAT ацетилтрансферазы", "Histone acetylation (write)"),
 ("PF00439","Bromodomain", "Бромодомен - чтение ацетил-лизина", "Histone acetyl (read)"),
 ("PF00385","Chromo","Хромодомен - чтение метил-лизина (HP1)", "Histone methyl (read)"),
 ("PF00628","PHD", "PHD-палец - чтение метилирования", "Histone (read)"),
 ("PF00567","Tudor", "Tudor-домен - чтение метил-остатков", "Histone (read)"),
 ("PF00125","Histone", "Кор-гистоны / гистон-подобные H2A/B/H3/H4","Histone fold"),
 ("PF00176","SNF2_N", "SNF2 АТФазы (SWI/SNF, ISWI, CHD)", "Chromatin remodeling"),
 ("PF02463", "SMC_N", "SMC-белки (когезин/конденсин)", "Cohesin/Condensin"),
 ("PF05434", "Rad21", "Rad21/Scc1 (когезин)", "Cohesin"),
]

In [14]:
fam_df = pd.DataFrame(FAMILIES, columns=["Pfam","Имя","Описание","Категория"])
ACCS = " ".join(f[0] for f in FAMILIES)
print(f"Семейств: {len(FAMILIES)} (требовалось >=10)")
fam_df

Семейств: 18 (требовалось >=10)


,Pfam,Имя,Описание,Категория
0,PF00145,DNA_methylase,ДНК-метилтрансферазы C5 (DNMT1/3),DNA methylation (write)
1,PF01429,MBD,Метил-CpG-связывающий домен (MeCP2/MBD),DNA methylation (read)
2,PF02008,zf-CXXC,CXXC цинк-палец (связывание неметил-CpG),DNA/CpG binding
3,PF12851,Tet_JBP,TET/JBP диоксигеназы (деметилирование),DNA demethylation
4,PF00856,SET,SET гистон-метилтрансферазы (EZH2/SETD),Histone methylation (write)
5,PF02373,JmjC,JmjC гистон-деметилазы (KDM),Histone demethylation (erase)
6,PF00850,Hist_deacetyl,Гистон-деацетилазы HDAC I/II,Histone deacetylation (erase)
7,PF02146,SIR2,Сиртуины - HDAC класс III,Histone deacetylation (erase)
8,PF01853,MOZ_SAS,MYST гистон-ацетилтрансферазы,Histone acetylation (write)
9,PF00583,Acetyltransf_1,GNAT ацетилтрансферазы,Histone acetylation (write)


Скачиваем HMM-профили семейств из InterPro и собираем в `hmm/epi.hmm`

In [15]:
!mkdir -p hmm
!rm -f hmm/epi.hmm
for acc in ACCS.split():
    !curl -sL "https://www.ebi.ac.uk/interpro/api/entry/pfam/{acc}?annotation=hmm" | gunzip >> hmm/epi.hmm
!grep -c '^NAME' hmm/epi.hmm

18


hmmsearch: профили семейств (query) против протеома (db); порог — gathering (`--cut_ga`)

In [16]:
!hmmsearch --cut_ga --noali --domtblout epi_hits.domtbl hmm/epi.hmm {PROT} > /dev/null
!echo "готово: $(grep -vc '^#' epi_hits.domtbl) строк попаданий"

готово: 383 строк попаданий


In [17]:
hits = []
with open("epi_hits.domtbl") as f:
    for line in f:
        if line.startswith("#"):
            continue
        c = line.split()
        if len(c) < 13: continue
        # c[3]=имя HMM-модели, c[4]=Pfam-accession (PFxxxxx.вер), c[6]=E-value
        hits.append((c[0], c[3], c[4].split(".")[0], float(c[6])))
hits_df = pd.DataFrame(hits, columns=["protein","family","acc","evalue"]).drop_duplicates(["protein","acc"])
print(f"попаданий белок-семейство: {len(hits_df)}; уникальных белков: {hits_df.protein.nunique()}")
hits_df.family.value_counts()

попаданий белок-семейство: 353; уникальных белков: 333


,count
family,
SET,90
Histone,65
Chromo,37
JmjC,27
Acetyltransf_1,27
PHD,26
SNF2-rel_dom,20
SMC_N,18
Bromodomain,13


Сопоставление белок-ген через GTF.

Белки в NCBI-аннотации именуются через `protein_id` в поле атрибутов GTF строки `CDS`.
Поэтому: белок (`protein_id`) -> транскрипт (`transcript_id`) -> ген (`gene_id`).


In [18]:
def parse_attrs_gtf(s):
    """Парсит поле атрибутов GTF (кавычки вокруг значений, разделитель '; ')."""
    attrs = {}
    for part in s.split(";"):
        part = part.strip()
        if " " in part:
            key, val = part.split(" ", 1)
            attrs[key] = val.strip().strip('"')
    return attrs

gene_name, gene_coords, prot2gene = {}, {}, {}
with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue
        c = line.rstrip("\n").split("\t")
        if len(c) < 9:
            continue
        ftype, attr = c[2], parse_attrs_gtf(c[8])
        if ftype == "gene":
            gid = attr.get("gene_id")
            gene_name[gid]   = attr.get("gene", gid)
            gene_coords[gid] = f"{c[0]}:{c[3]}-{c[4]}({c[6]})"
        elif ftype == "CDS":
            pid = attr.get("protein_id")
            gid = attr.get("gene_id")
            if pid and gid:
                prot2gene[pid] = gid

def protein_to_gene(p):
    g = prot2gene.get(p)
    return g, gene_name.get(g, g)

hits_df[["gene_id","gene"]] = hits_df.protein.apply(lambda p: pd.Series(protein_to_gene(p)))
print("белков сопоставлено с генами:", hits_df.gene_id.notna().sum(), "/", len(hits_df))
hits_df.head()

белков сопоставлено с генами: 353 / 353


,protein,family,acc,evalue,gene_id,gene
0,CAL2038045.1,MBD,PF01429,7.600000e-22,CAEBRE_10808,CAEBRE_10808
1,CAL2037545.1,MBD,PF01429,9.800000e-15,CAEBRE_10307,CAEBRE_10307
2,CAL2037350.1,SET,PF00856,8.800000e-27,CAEBRE_10112,CAEBRE_10112
3,CAL2032091.1,SET,PF00856,7.800000e-26,CAEBRE_04837,CAEBRE_04837
4,CAL2036203.1,SET,PF00856,3.600000e-25,CAEBRE_08963,CAEBRE_08963


Итоговая таблица «Проверяемое семейство — Ген — Координаты»

In [19]:
acc2info = {acc:(name,desc,cat) for acc,name,desc,cat in FAMILIES}  # ключ — Pfam accession
rows = []
for _, r in hits_df.iterrows():
    name, desc, cat = acc2info.get(r.acc, (r.family, "", ""))
    rows.append({
        "Проверяемое семейство": f"{r.acc} {name} - {desc}",
        "Категория": cat,
        "Название гена": r.gene if pd.notna(r.gene) else r.protein,
        "Белок": r.protein,
        "Координаты гена": gene_coords.get(r.gene_id, ""),
    })
epi_table = (pd.DataFrame(rows)
             .sort_values(["Категория","Проверяемое семейство","Название гена"])
             .reset_index(drop=True))
epi_table.to_csv("epigenetic_genes.csv", index=False)

print("Генов по эпигенетическим семействам:")
print(epi_table.groupby("Проверяемое семейство")["Название гена"].nunique().sort_values(ascending=False))
print("\nВсего уникальных генов:", epi_table["Название гена"].nunique())
epi_table.head(30)

Генов по эпигенетическим семействам:
Проверяемое семейство
PF00856 SET - SET гистон-метилтрансферазы (EZH2/SETD)          90
PF00125 Histone - Кор-гистоны / гистон-подобные H2A/B/H3/H4    65
PF00385 Chromo - Хромодомен - чтение метил-лизина (HP1)        37
PF02373 JmjC - JmjC гистон-деметилазы (KDM)                    27
PF00583 Acetyltransf_1 - GNAT ацетилтрансферазы                27
PF00628 PHD - PHD-палец - чтение метилирования                 26
PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)              20
PF02463 SMC_N - SMC-белки (когезин/конденсин)                  18
PF00439 Bromodomain - Бромодомен - чтение ацетил-лизина        13
PF00567 Tudor - Tudor-домен - чтение метил-остатков             9
PF00850 Hist_deacetyl - Гистон-деацетилазы HDAC I/II            9
PF01853 MOZ_SAS - MYST гистон-ацетилтрансферазы                 6
PF02146 SIR2 - Сиртуины - HDAC класс III                        3
PF01429 MBD - Метил-CpG-связывающий домен (MeCP2/MBD)           2
PF05434 Rad21 - R

,Проверяемое семейство,Категория,Название гена,Белок,Координаты гена
0,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_01914,CAL2029182.1,OZ038406.1:7822363-7826583(+)
1,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_02468,CAL2029736.1,OZ038406.1:9960443-9966844(+)
2,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_02672,CAL2029940.1,OZ038406.1:10833516-10844477(+)
3,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_03168,CAL2030434.1,OZ038406.1:13979186-13988172(-)
4,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_05745,CAL2032997.1,OZ038407.1:9847165-9851205(-)
5,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_05877,CAL2033128.1,OZ038407.1:10401373-10407246(-)
6,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_08342,CAL2035587.1,OZ038408.1:2240340-2252675(-)
7,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_08831,CAL2036072.1,OZ038408.1:4977530-4988703(+)
8,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_09332,CAL2036572.1,OZ038408.1:7371609-7375132(-)
9,"PF00176 SNF2_N - SNF2 АТФазы (SWI/SNF, ISWI, CHD)",Chromatin remodeling,CAEBRE_09389,CAL2036629.1,OZ038408.1:7609372-7613537(+)


## 4. G-квадруплексы

Паттерн: `(G{3,5}[ATGC]{1,7}){3,}G{3,5}`.
- ищем на обоих стрендах: «+» — по G-паттерну, «−» — по комплементарному
  C-паттерну `(C{3,5}[ATGC]{1,7}){3,}C{3,5}` на той же (плюс-)цепи;
- последовательность в верхнем регистре (учёт регистра).

Результат — `g4.bed` (BED6, координаты 0-based).

In [20]:
g4_plus  = re.compile(r"(G{3,5}[ATGC]{1,7}){3,}G{3,5}")
g4_minus = re.compile(r"(C{3,5}[ATGC]{1,7}){3,}C{3,5}")

In [21]:
g4_rows = []
for chrom in WORK_CHROMS:
    seq = genome[chrom]
    for strand, rx in (("+", g4_plus), ("-", g4_minus)):
        for m in rx.finditer(seq):
            g4_rows.append((chrom, m.start(), m.end(), strand))

with open("g4.bed", "w") as f:
    for i,(c,s,e,st) in enumerate(g4_rows):
        f.write(f"{c}\t{s}\t{e}\tG4_{i:06d}\t{e-s}\t{st}\n")

In [22]:
work_len = sum(CHROM_SIZES[c] for c in WORK_CHROMS)
print(f"G-квадруплексов всего: {len(g4_rows):,}")
print("  по стрендам:", Counter(r[3] for r in g4_rows))
print("  плотность: %.1f на Мб" % (len(g4_rows)/(work_len/1e6)))

G-квадруплексов всего: 5,093
  по стрендам: Counter({'+': 2549, '-': 2544})
  плотность: 40.3 на Мб


## 5. Z-ДНК — `zhunt` (фильтр z-score > 400)

Компилируем `zhun3.c` (Ho Lab). Запуск: `zhunt windowsize minsize maxsize file`.

**Формат `<file>.Z-SCORE`:** шапка `имя seqlen fromdin todin`, далее по строке на нуклеотид
(строка j = позиция j, 0-based), 4 колонки: `bestdl  slope  Z-Hunt-score  antisyn`.
Третья колонка — **Z-Hunt score**; фильтруем её по **> 400** (порог из методики ZDNABERT,
Beknazarov et al. 2020).

`zhunt` после расчёта открывает интерактивное меню → запускаем со вводом из `/dev/null`.
Параметры (для отчёта): **windowsize=12, min=8, max=12**. Считаем по `WORK_CHROMS`.

In [ ]:
%%bash
mkdir -p zhunt
[ -x zhunt/zhunt ] && { echo "zhunt уже собран"; exit 0; }
curl -sL https://raw.githubusercontent.com/Ho-Lab-Colostate/zhunt/master/zhun3.c -o zhunt/zhun3.c
cc -O2 -U_FORTIFY_SOURCE -D_FORTIFY_SOURCE=0 -fno-stack-protector -o zhunt/zhunt zhunt/zhun3.c -lm
echo "compiled: $([ -x zhunt/zhunt ] && echo yes || echo no)"

готовим по одному файлу-последовательности на хромосому (N→A, координаты 1:1)

In [ ]:
CHROMS = WORK_CHROMS
for chrom in CHROMS:
    safe = re.sub(r"[^ACGT]", "A", genome[chrom])
    (Path("zhunt")/f"{chrom}.seq").write_text(safe + "\n")
print("подготовлено .seq:", len(CHROMS))

запуск zhunt по каждой хромосоме (последовательно; долго — ~18 мин на 29 Мб)

In [ ]:
for chrom in CHROMS:
    z = Path("zhunt")/f"{chrom}.seq.Z-SCORE"
    if z.exists() and z.stat().st_size > 0:
        print("zhunt пропуск (есть):", chrom); continue
    print("zhunt:", chrom)
    !cd zhunt && ./zhunt 12 8 12 "{chrom}.seq" < /dev/null > /dev/null 2>&1

zhunt: OZ038406.1
zhunt: OZ038407.1
zhunt: OZ038408.1
zhunt: OZ038409.1
zhunt: OZ038410.1
zhunt: OZ038411.1
zhunt: CAXIWG010000001.1
zhunt: CAXIWG010000002.1
zhunt: CAXIWG010000003.1
zhunt: CAXIWG010000004.1
zhunt: CAXIWG010000005.1
zhunt: CAXIWG010000006.1
zhunt: CAXIWG010000007.1
zhunt: CAXIWG010000008.1
zhunt: CAXIWG010000009.1
zhunt: CAXIWG010000010.1
zhunt: CAXIWG010000011.1
zhunt: CAXIWG010000012.1
zhunt: CAXIWG010000013.1
zhunt: CAXIWG010000014.1
zhunt: CAXIWG010000015.1
zhunt: CAXIWG010000016.1
zhunt: CAXIWG010000017.1
zhunt: CAXIWG010000018.1
zhunt: CAXIWG010000019.1
zhunt: CAXIWG010000020.1
zhunt: CAXIWG010000021.1
zhunt: CAXIWG010000022.1
zhunt: CAXIWG010000023.1
zhunt: CAXIWG010000024.1
zhunt: CAXIWG010000025.1
zhunt: CAXIWG010000026.1
zhunt: CAXIWG010000027.1
zhunt: CAXIWG010000028.1
zhunt: CAXIWG010000029.1
zhunt: CAXIWG010000030.1
zhunt: CAXIWG010000031.1
zhunt: CAXIWG010000032.1
zhunt: CAXIWG010000033.1
zhunt: CAXIWG010000034.1
zhunt: CAXIWG010000035.1
zhunt: CAXIWG0100

фильтр Z-Hunt score > 400, координаты в `_zdna_raw.bed`

In [ ]:
ZSCORE_MIN = 400
with open("_zdna_raw.bed", "w") as bed:
    for chrom in CHROMS:
        zfile = Path("zhunt")/f"{chrom}.seq.Z-SCORE"
        if not zfile.exists():
            print("нет вывода для", chrom); continue
        n = 0
        with open(zfile) as f:
            f.readline()
            for j, line in enumerate(f):
                c = line.split()
                if len(c) >= 3 and float(c[2]) > ZSCORE_MIN:
                    bed.write(f"{chrom}\t{j}\t{j+2}\t{c[2]}\n")
                    n += 1
        print(f"  {chrom}: позиций z>{ZSCORE_MIN} = {n:,}")

  OZ038406.1: позиций z>400 = 20,418
  OZ038407.1: позиций z>400 = 17,755
  OZ038408.1: позиций z>400 = 18,234
  OZ038409.1: позиций z>400 = 22,232
  OZ038410.1: позиций z>400 = 19,470
  OZ038411.1: позиций z>400 = 29,245
  CAXIWG010000001.1: позиций z>400 = 8
  CAXIWG010000002.1: позиций z>400 = 1
  CAXIWG010000003.1: позиций z>400 = 4
  CAXIWG010000004.1: позиций z>400 = 0
  CAXIWG010000005.1: позиций z>400 = 234
  CAXIWG010000006.1: позиций z>400 = 34
  CAXIWG010000007.1: позиций z>400 = 12
  CAXIWG010000008.1: позиций z>400 = 13
  CAXIWG010000009.1: позиций z>400 = 80
  CAXIWG010000010.1: позиций z>400 = 34
  CAXIWG010000011.1: позиций z>400 = 11
  CAXIWG010000012.1: позиций z>400 = 0
  CAXIWG010000013.1: позиций z>400 = 0
  CAXIWG010000014.1: позиций z>400 = 5
  CAXIWG010000015.1: позиций z>400 = 0
  CAXIWG010000016.1: позиций z>400 = 15
  CAXIWG010000017.1: позиций z>400 = 77
  CAXIWG010000018.1: позиций z>400 = 22
  CAXIWG010000019.1: позиций z>400 = 19
  CAXIWG010000020.1: пози

объединяем соседние позиции в Z-ДНК-участки (score = макс. z-score)

In [ ]:
!sort -k1,1 -k2,2n _zdna_raw.bed | bedtools merge -c 4 -o max | awk 'BEGIN{OFS="\t"}{print $1,$2,$3,"Z_"NR,$4,"+"}' > zdna_zhunt.bed
!echo "Z-ДНК участков: $(wc -l < zdna_zhunt.bed)"

Z-ДНК участков: 17989


### ZDNABERT

In [ ]:
!pip install -q transformers
import torch
from transformers import BertTokenizer, BertForTokenClassification
import numpy as np
import scipy.ndimage
from tqdm import tqdm

In [ ]:
def seq2kmer(seq, k=6):
    """Разбивает последовательность на k-меры (перекрывающиеся окна)."""
    return [seq[x:x+k] for x in range(len(seq) + 1 - k)]

def split_seq(seq, length=512, pad=16):
    """
    Разбивает длинную последовательность на куски для батч-обработки.
    pad — перекрытие между кусками, чтобы не терять предсказания на стыках.
    """
    res = []
    for st in range(0, len(seq), length - pad):
        end = min(st + length, len(seq))
        res.append(seq[st:end])
    return res

def stitch_np_seq(np_seqs, pad=16):
    """Собирает предсказания обратно в один массив, удаляя перекрытия."""
    res = np.array([])
    for seq in np_seqs:
        res = res[:-pad]
        res = np.concatenate([res, seq])
    return res

In [ ]:
MODEL_NAME = 'HG kouzine'
model_confidence_threshold = 0.5 # порог вероятности для положительного предсказания
minimum_sequence_length = 10 # минимальная длина предсказанного региона

model_ids = {
    'HG chipseq': '1VAsp8I904y_J0PUhAQqpSlCn1IqfG0FB',
    'HG kouzine': '1dAeAt5Gu2cadwDhbc7OnenUgDLHlUvkx',
    'MM chipseq': '1W6GEgHNoitlB-xXJbLJ_jDW4BF35W1Sd',
    'MM kouzine': '1dXpQFmheClKXIEoqcZ7kgCwx6hzVCv3H',
}
model_id = model_ids[MODEL_NAME]

In [ ]:
!gdown {model_id}
!gdown 10sF8Ywktd96HqAL0CwvlZZUUGj05CGk5  # config.json
!gdown 16bT7HDv71aRwyh3gBUbKwign1mtyLD2d  # special_tokens_map.json
!gdown 1EE9goZ2JRSD8UTx501q71lGCk-CK3kqG  # tokenizer_config.json
!gdown 1gZZdtAoDnDiLQqjQfGyuwt268Pe5sXW0  # vocab.txt

!mkdir -p model
!mv pytorch_model.bin config.json special_tokens_map.json tokenizer_config.json vocab.txt model/

In [ ]:
tokenizer = BertTokenizer.from_pretrained('model/')
zdnabert_model = BertForTokenClassification.from_pretrained('model/')
# Используем GPU если доступен, иначе CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
zdnabert_model = zdnabert_model.to(device)
print(f"Модель загружена, устройство: {device}")

In [ ]:
out = []
input_file = 'genome.fna'

torch.backends.cudnn.benchmark = True

for seq_record in SeqIO.parse(input_file, 'fasta'):
    kmer_seq = seq2kmer(str(seq_record.seq).upper(), 6)
    seq_pieces = split_seq(kmer_seq, length=512, pad=16)
    print(f"Обрабатываем: {seq_record.name} (длина {len(seq_record.seq)} нт)")

    with torch.no_grad():
        preds = []
        batch_size = 4
        for i in tqdm(range(0, len(seq_pieces), batch_size), desc=f"  {seq_record.name}"):
            batch = seq_pieces[i:i+batch_size]

            encoded = tokenizer(
                [' '.join(p) for p in batch],
                add_special_tokens=False,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512
            )
            input_ids = encoded['input_ids'].to(device)

            logits = zdnabert_model(input_ids)[0]
            probs = torch.softmax(logits, dim=-1)[:, :, 1].cpu().numpy()
            preds.extend(probs)

    full_pred = stitch_np_seq(preds)
    labeled, n_labels = scipy.ndimage.label(full_pred > model_confidence_threshold)

    for label in range(1, n_labels + 1):
        candidate = np.where(labeled == label)[0]
        if candidate.shape[0] >= minimum_sequence_length:
            out.append((seq_record.name, int(candidate[0]), int(candidate[-1]) + 5))

print(f"\nВсего предсказано Z-ДНК регионов: {len(out)}")

In [ ]:
# Сохраняем предсказания ZDNABERT в BED
os.makedirs("ZDNAbert", exist_ok=True)
with open('ZDNAbert/ZDNAbert.bed', 'w') as f:
    for chrom, start, end in out:
        f.write(f"{chrom}\t{start}\t{end}\n")

!echo "ZDNAbert.bed создан, первые строки:"
!head ZDNAbert/ZDNAbert.bed

In [ ]:
out = []

with open('ZDNAbert.bed', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split('\t')
            if len(parts) >= 3:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                out.append((chrom, start, end))

print(f"Загружено из ZDNAbert.bed: {len(out)} записей")

with open('ZDNAbert_raw.bed', 'w') as f:
    for chrom, start, end in out:
        f.write(f"{chrom}\t{start}\t{end}\n")

!sort -k1,1 -k2,2n ZDNAbert_raw.bed > ZDNAbert_sorted.bed
!bedtools merge -i ZDNAbert_sorted.bed -d 1 > ZDNAbert_merged.bed

merged_out = []
with open('ZDNAbert_merged.bed') as f:
    for line in f:
        chrom, start, end = line.strip().split('\t')[:3]
        merged_out.append((chrom, int(start), int(end)))

print(f"Было (raw): {len(out):,}")
print(f"Стало (merged): {len(merged_out):,}")

with open('ZDNAbert.bed', 'w') as f:
    for chrom, start, end in merged_out:
        f.write(f"{chrom}\t{start}\t{end}\n")

os.makedirs("ZDNAbert", exist_ok=True)
with open('ZDNAbert/ZDNAbert.bed', 'w') as f:
    for chrom, start, end in merged_out:
        f.write(f"{chrom}\t{start}\t{end}\n")

!head -5 ZDNAbert.bed

## 6. Распределение структур по геному

In [29]:
# Извлекаем координаты из GTF NCBI
# В NCBI GTF есть признак "exon" напрямую
genes, exons_by_rna = [], defaultdict(list)

with open(GTF) as f:
    for line in f:
        if line.startswith("#"): continue
        c = line.rstrip("\n").split("\t")
        if len(c) < 9: continue
        seqid, ftype, start, end, strand = c[0], c[2], int(c[3]), int(c[4]), c[6]
        # Работаем только с рабочими хромосомами
        if seqid not in WORK_CHROMS: continue
        attr = parse_attrs_gtf(c[8])
        if ftype == "gene":
            genes.append((seqid, start-1, end, strand))
        elif ftype == "exon":
            tx = attr.get("transcript_id", "")
            exons_by_rna[tx].append((seqid, start-1, end, strand))

def write_bed(path, intervals):
    with open(path, "w") as f:
        for iv in intervals:
            chrom, s, e = iv[0], max(0, iv[1]), min(iv[2], CHROM_SIZES.get(iv[0], iv[2]))
            if e > s:
                f.write(f"{chrom}\t{s}\t{e}\n")

Path("features").mkdir(exist_ok=True)
write_bed("features/exons.bed", [e for lst in exons_by_rna.values() for e in lst])

introns = []
for lst in exons_by_rna.values():
    s = sorted(lst, key=lambda x: x[1])
    for a, b in zip(s, s[1:]):
        if b[1] > a[2]:
            introns.append((a[0], a[2], b[1]))
write_bed("features/introns.bed", introns)

promoters, downstream, gene_iv = [], [], []
for chrom, s, e, strand in genes:
    gene_iv.append((chrom, s, e))
    if strand == "+":
        promoters.append((chrom, s-1000, s)); downstream.append((chrom, e, e+200))
    else:
        promoters.append((chrom, e, e+1000)); downstream.append((chrom, s-200, s))
write_bed("features/promoters.bed", promoters)
write_bed("features/downstream.bed", downstream)
write_bed("features/genes.bed", gene_iv)
print("BED участков записаны в features/")

BED участков записаны в features/


In [30]:
for name in ["exons","introns","promoters","downstream","genes"]:
    !sort -k1,1 -k2,2n features/{name}.bed | bedtools merge -i - > features/{name}.merged
    !mv features/{name}.merged features/{name}.bed
!sort -k1,1 -k2,2n features/genes.bed | bedtools complement -i - -g {SIZES} > features/intergenic.bed

for name in ["exons","introns","promoters","downstream","intergenic"]:
    n = sum(1 for _ in open(f"features/{name}.bed"))
    print(f"{name:12} участков: {n:>8,}")

exons        участков:  143,773
introns      участков:  117,973
promoters    участков:   21,346
downstream   участков:   23,316
intergenic   участков:   24,691


In [31]:
# приоритет отнесения структуры к одному участку
PRIORITY = [("Exons","exons"), ("Introns","introns"),
            ("Promoters (1000 up)","promoters"), ("Downstream (200 bp)","downstream"),
            ("Intergenic","intergenic")]

def classify(struct_bed):
    "Каждую структуру относим к первому по приоритету участку, который она пересекает."
    assigned = Counter()
    !sort -k1,1 -k2,2n {struct_bed} > _remaining.bed
    total_s = sum(1 for _ in open("_remaining.bed"))
    for label, key in PRIORITY:
        feat = f"features/{key}.bed"
        hit = !bedtools intersect -u -a _remaining.bed -b {feat}
        !bedtools intersect -v -a _remaining.bed -b {feat} > _rest.bed
        assigned[label] = len(hit)
        !mv _rest.bed _remaining.bed
    return assigned, total_s

def regions_with_structure(struct_bed):
    "Сколько участков каждого типа содержат >=1 структуру."
    res = {}
    for label, key in PRIORITY:
        feat = f"features/{key}.bed"
        n_tot = sum(1 for _ in open(feat))
        hit = !bedtools intersect -u -a {feat} -b {struct_bed}
        res[label] = (len(hit), n_tot)
    return res

In [32]:
# Таблица 1: число и доля структур по участкам
def table1(struct_bed, name):
    counts, tot = classify(struct_bed)
    return pd.DataFrame({
        "Участок":[l for l,_ in PRIORITY],
        f"Число ({name})":[counts[l] for l,_ in PRIORITY],
        f"Доля ({name})":[round(counts[l]/tot,4) if tot else 0 for l,_ in PRIORITY]})

TABLE1 = (table1("g4.bed","G4")
          .merge(table1("zdna_zhunt.bed","Zhunt"), on="Участок")
          .merge(table1("ZDNAbert.bed","ZDNABERT"), on="Участок"))
TABLE1.to_csv("table1_structures_distribution.csv", index=False)
print("Таблица 1: распределение структур по участкам")
TABLE1


Таблица 1: распределение структур по участкам


,Участок,Число (G4),Доля (G4),Число (Zhunt),Доля (Zhunt),Число (ZDNABERT),Доля (ZDNABERT)
0,Exons,362,0.0711,3359,0.1867,2768,0.4017
1,Introns,1598,0.3138,4468,0.2484,1213,0.1761
2,Promoters (1000 up),808,0.1586,3078,0.1711,739,0.1073
3,Downstream (200 bp),104,0.0204,305,0.0170,88,0.0128
4,Intergenic,2221,0.4361,6779,0.3768,2082,0.3022


In [33]:
# Таблица 2: число и доля участков, содержащих структуру
def table2(struct_bed, name):
    res = regions_with_structure(struct_bed)
    return pd.DataFrame({
        "Участок":[l for l,_ in PRIORITY],
        f"Участков со структурой ({name})":[res[l][0] for l,_ in PRIORITY],
        f"Доля участков ({name})":[round(res[l][0]/res[l][1],4) if res[l][1] else 0 for l,_ in PRIORITY]})

TABLE2 = (table2("g4.bed","G4")
          .merge(table2("zdna_zhunt.bed","Zhunt"), on="Участок")
          .merge(table2("ZDNAbert.bed","ZDNABERT"), on="Участок"))
TABLE2.to_csv("table2_regions_with_structure.csv", index=False)
print("Таблица 2: доля участков с >=1 структурой")
TABLE2


Таблица 2: доля участков с >=1 структурой


,Участок,Участков со структурой (G4),Доля участков (G4),Участков со структурой (Zhunt),Доля участков (Zhunt),Участков со структурой (ZDNABERT),Доля участков (ZDNABERT)
0,Exons,341,0.0024,2935,0.0204,1770,0.0123
1,Introns,1165,0.0099,3195,0.0271,1424,0.0121
2,Promoters (1000 up),773,0.0362,3044,0.1426,692,0.0324
3,Downstream (200 bp),156,0.0067,539,0.0231,205,0.0088
4,Intergenic,1848,0.0748,5204,0.2108,1387,0.0562


## 7. Сравнение с фоном

`bedtools shuffle` случайно перемещает структуры по геному (сохраняя число и длины);
классифицируем так же и усредняем по нескольким перестановкам.
**Обогащение** = доля наблюдаемая / доля фоновая (>1 — структуры в участке чаще случайного).

In [34]:
def enrichment_table(struct_bed, name, n_iter=5):
    obs, tot = classify(struct_bed)
    bg = Counter()
    for it in range(n_iter):
        seed = it+1
        !bedtools shuffle -i {struct_bed} -g {SIZES} -seed {seed} -noOverlapping 2>/dev/null | sort -k1,1 -k2,2n > _shuf.bed
        counts, t = classify("_shuf.bed")
        for l,_ in PRIORITY:
            bg[l] += (counts[l]/t)/n_iter if t else 0
    rows = []
    for l,_ in PRIORITY:
        o = obs[l]/tot if tot else 0
        rows.append({"Участок":l, f"Доля набл. ({name})":round(o,4),
                     f"Доля фон ({name})":round(bg[l],4),
                     f"Обогащение ({name})":round(o/bg[l],2) if bg[l] else float('nan')})
    return pd.DataFrame(rows)

ENRICH = (enrichment_table("g4.bed","G4")
          .merge(enrichment_table("zdna_zhunt.bed","Zhunt"), on="Участок")
          .merge(enrichment_table("ZDNAbert.bed","ZDNABERT"), on="Участок"))
ENRICH.to_csv("table1_vs_background.csv", index=False)
print("Сравнение распределения структур с фоном")
ENRICH


Сравнение распределения структур с фоном


,Участок,Доля набл. (G4),Доля фон (G4),Обогащение (G4),Доля набл. (Zhunt),Доля фон (Zhunt),Обогащение (Zhunt),Доля набл. (ZDNABERT),Доля фон (ZDNABERT),Обогащение (ZDNABERT)
0,Exons,0.0711,0.2989,0.24,0.1867,0.2841,0.66,0.4017,0.3391,1.18
1,Introns,0.3138,0.2002,1.57,0.2484,0.2172,1.14,0.1761,0.1762,1.00
2,Promoters (1000 up),0.1586,0.1329,1.19,0.1711,0.1305,1.31,0.1073,0.1237,0.87
3,Downstream (200 bp),0.0204,0.0196,1.04,0.0170,0.0200,0.85,0.0128,0.0185,0.69
4,Intergenic,0.4361,0.3484,1.25,0.3768,0.3482,1.08,0.3022,0.3424,0.88


## 8. Итоговые файлы для GitHub / README

- `data/genome.fna` — геном (`GCA_964036135.1_CAEBRE_CFB2252`)
- `data/annotation.gtf` — аннотация NCBI
- `data/proteome.faa` — протеом NCBI
- `g4.bed` — G-квадруплексы
- `zdna_zhunt.bed` — Z-ДНК (zhunt)
- `epigenetic_genes.csv` — таблица «семейство–ген»
- `table1_structures_distribution.csv`, `table2_regions_with_structure.csv`, `table1_vs_background.csv`


In [ ]:
# Итого для отчёта - GCA_964036135.1 (Caenorhabditis brenneri)
print("="*60)
print(f"Длина генома: {total:,} bp,  GC {100*gc/(total-nN):.1f}%,  N50 {n50:,}")
print(f"Режим: {'ТЕСТ ('+','.join(WORK_CHROMS)+')' if TEST_MODE else 'ПОЛНЫЙ ГЕНОМ'}")
print(f"Генов (GTF NCBI): {n_genes:,}")
print(f"Белков (протеом NCBI): {n_prot:,}")
print(f"Эпигенетич. генов: {epi_table['Название гена'].nunique()} по {epi_table['Проверяемое семейство'].nunique()} семействам")
print(f"G-квадруплексов: {len(g4_rows):,}")
for p in ["data/annotation.gtf","data/proteome.faa","g4.bed","zdna_zhunt.bed","epigenetic_genes.csv",
          "table1_structures_distribution.csv","table2_regions_with_structure.csv",
          "table1_vs_background.csv"]:
    print(("  OK  " if Path(p).exists() else "  --  ")+p)

Длина генома: 126,489,859 bp,  GC 38.4%,  N50 20,836,669
Режим: ПОЛНЫЙ ГЕНОМ
Генов (GTF NCBI): 25,612
Белков (протеом NCBI): 25,588
Эпигенетич. генов: 314 по 13 семействам
G-квадруплексов: 5,093
  OK  data/annotation.gtf
  OK  data/proteome.faa
  OK  g4.bed
  OK  zdna_zhunt.bed
  OK  epigenetic_genes.csv
  OK  table1_structures_distribution.csv
  OK  table2_regions_with_structure.csv
  OK  table1_vs_background.csv
